# FMD Terminal Extrapolation Experiment

Evaluate the branch experiment that extends tblastn mat_peptide boundaries when HSP query coordinates show small missing N/C-terminal amino acids.

Run this notebook on branch `experiment/fmd-terminal-extrapolation-v2`. Outputs go to `terminal_extrapolation_outputs/`.

## What This Compares

- **Baseline**: existing strict-clean FMD outputs generated before terminal extrapolation.
- **Experiment**: reruns FMD strict-clean tblastn using the current branch code.

Main questions:

- Did exact/coordinate accuracy improve?
- Did boundary-offset tool cases decrease?
- Did any exact matches regress?

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'app').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from app.validation._shared.validation_utils import *

TIER = 'strict_clean'
BASE_DIR = Path.cwd()
if not (BASE_DIR / 'inputs').exists() and (BASE_DIR / 'tiers' / TIER / 'inputs').exists():
    BASE_DIR = BASE_DIR / 'tiers' / TIER

INPUT_DIR = BASE_DIR / 'inputs'
BASELINE_DIR = BASE_DIR / 'outputs'
OUTPUT_DIR = BASE_DIR / 'terminal_extrapolation_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 10})


## Baseline Snapshot

This reads the previous strict-clean FMD results. It does not rerun BLAST.

In [ ]:
baseline_vs_truth = pd.read_csv(BASELINE_DIR / 'tblastn_vs_truth.tsv', sep='	')
baseline_fmd_df = baseline_vs_truth[baseline_vs_truth['virus'] == 'FMD'].copy()
baseline_fmd_df.head()

## Run Experiment On FMD Strict-Clean

This reruns only FMD. It will call tblastn, so this is the slow cell.

In [ ]:
experiment_df, experiment_summary = run_tblastn_against_truth(
    virus_label='FMD',
    ref_path=DATA / 'FMD_ref_test.gb',
    query_path=INPUT_DIR / 'FMD_records.gb',
    output_dir=OUTPUT_DIR / 'fmd',
    progress=True,
)

experiment_df.to_csv(OUTPUT_DIR / 'fmd_tblastn_vs_truth.tsv', sep='	', index=False)
experiment_summary.to_csv(OUTPUT_DIR / 'fmd_tblastn_summary.tsv', sep='	', index=False)
experiment_summary

## Raw Accuracy Comparison

In [ ]:
def raw_fmd_summary(df, label):
    total = len(df)
    exact = int(df['exact_match'].sum())
    coord = int(df['coord_correct'].sum())
    return {
        'run': label,
        'total': total,
        'exact_match': exact,
        'coord_correct': coord,
        'coord_only': coord - exact,
        'failed_coord_or_name': total - coord,
        'exact_pct': round(exact / total * 100, 2),
        'coord_pct': round(coord / total * 100, 2),
        'failed_pct': round((total - coord) / total * 100, 2),
    }

raw_compare = pd.DataFrame([
    raw_fmd_summary(baseline_fmd_df, 'baseline'),
    raw_fmd_summary(experiment_df, 'terminal_extrapolation'),
])
raw_compare.to_csv(OUTPUT_DIR / 'fmd_raw_accuracy_comparison.tsv', sep='	', index=False)
raw_compare

In [ ]:
plot_df = raw_compare.set_index('run')[['exact_pct', 'coord_pct']]
ax = plot_df.plot(kind='bar', figsize=(7, 4), color=['#2f7d4f', '#4c78a8'])
ax.set_ylim(0, 100)
ax.set_ylabel('Percent')
ax.set_xlabel('')
ax.set_title('FMD strict-clean accuracy: baseline vs terminal extrapolation')
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3, fontsize=9)
fig = ax.figure
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'fmd_accuracy_comparison.png', dpi=180, bbox_inches='tight')


## Ref-Aware Failure Adjudication

Reuses the FMD final adjudication logic from the strict-clean notebook so causes are comparable before/after.

In [ ]:
fmd_bundle = load_reference_bundle(DATA / 'FMD_ref_test.gb')
fmd_ref_model = pd.DataFrame([
    {
        'ref_name': feature['name'],
        'ref_start': feature['start'],
        'ref_end': feature['end'],
        'ref_len': feature['end'] - feature['start'] + 1,
    }
    for feature in fmd_bundle['features']
])


def prepare_fmd_failures(df):
    failures = df[~df['exact_match']].copy()
    if failures.empty:
        return failures
    if 'ref_name' not in failures.columns:
        failures['ref_name'] = failures['pred_name']
    failures['delta_start'] = failures['pred_start'] - failures['truth_start']
    failures['delta_end'] = failures['pred_end'] - failures['truth_end']
    failures['pred_len'] = failures['pred_end'] - failures['pred_start'] + 1
    failures['truth_len'] = failures['truth_end'] - failures['truth_start'] + 1
    failures = failures.merge(fmd_ref_model, on='ref_name', how='left')
    failures['pred_minus_ref_len'] = failures['pred_len'] - failures['ref_len']
    failures['truth_minus_ref_len'] = failures['truth_len'] - failures['ref_len']
    failures['pred_minus_truth_len'] = failures['pred_len'] - failures['truth_len']
    return failures


def final_reason(row):
    feature = row['ref_name']
    mode = row.get('failure_mode')
    ds = row.get('delta_start')
    de = row.get('delta_end')
    pred_ref = row.get('pred_minus_ref_len')
    truth_ref = row.get('truth_minus_ref_len')

    if pd.isna(row.get('truth_name')):
        return pd.Series({'final_blame': 'ref_truth', 'final_cause': 'truth_feature_absent'})
    if pred_ref == 0 and truth_ref != 0:
        return pd.Series({'final_blame': 'ref_truth', 'final_cause': 'ref_query_boundary_convention_mismatch'})
    if mode == 'Not lifted':
        return pd.Series({'final_blame': 'tool', 'final_cause': 'no_hit'})
    if feature == '3A' and ds == 0 and de == -111:
        return pd.Series({'final_blame': 'tool', 'final_cause': 'major_c_terminal_truncation'})
    if feature in {'Lpro', '3Cpro'} and ds == 12 and de == 0:
        return pd.Series({'final_blame': 'tool', 'final_cause': 'n_terminal_truncation_12bp'})
    if feature == '2A' and mode == 'Boundary offset':
        return pd.Series({'final_blame': 'tool', 'final_cause': 'short_peptide_boundary_offset'})
    if feature == '3B' and de == -3:
        return pd.Series({'final_blame': 'tool', 'final_cause': 'minor_c_terminal_truncation_3bp'})
    if feature == 'VP1':
        return pd.Series({'final_blame': 'tool', 'final_cause': 'minor_vp1_boundary_offset'})
    return pd.Series({'final_blame': 'tool', 'final_cause': 'boundary_offset_matches_neither_ref_nor_truth'})


def adjudicate(df, label):
    failures = prepare_fmd_failures(df)
    if failures.empty:
        return failures, pd.DataFrame(), pd.DataFrame()
    final_cols = failures.apply(final_reason, axis=1)
    final = pd.concat([failures, final_cols], axis=1)
    final['run'] = label
    blame = final.groupby(['run', 'final_blame']).size().reset_index(name='cases')
    cause = final.groupby(['run', 'final_blame', 'final_cause']).size().reset_index(name='cases')
    return final, blame, cause

baseline_final_cases, baseline_blame_compare, baseline_cause_compare = adjudicate(baseline_fmd_df, 'baseline')
experiment_final_cases, experiment_blame_compare, experiment_cause_compare = adjudicate(experiment_df, 'terminal_extrapolation')

final_cases = pd.concat([baseline_final_cases, experiment_final_cases], ignore_index=True)
blame_compare = pd.concat([baseline_blame_compare, experiment_blame_compare], ignore_index=True)
cause_compare = pd.concat([baseline_cause_compare, experiment_cause_compare], ignore_index=True)

final_cases.to_csv(OUTPUT_DIR / 'fmd_final_cases_comparison.tsv', sep='	', index=False)
blame_compare.to_csv(OUTPUT_DIR / 'fmd_final_blame_comparison.tsv', sep='	', index=False)
cause_compare.to_csv(OUTPUT_DIR / 'fmd_final_cause_comparison.tsv', sep='	', index=False)

cause_compare

In [ ]:
pivot = cause_compare.pivot_table(
    index=['final_blame', 'final_cause'],
    columns='run',
    values='cases',
    fill_value=0,
).reset_index()
if 'baseline' in pivot.columns and 'terminal_extrapolation' in pivot.columns:
    pivot['delta'] = pivot['terminal_extrapolation'] - pivot['baseline']
pivot.to_csv(OUTPUT_DIR / 'fmd_final_cause_delta.tsv', sep='	', index=False)
pivot

## Cases Fixed Or Regressed

A fixed case was non-exact in baseline but exact after terminal extrapolation. A regressed case was exact in baseline but non-exact after terminal extrapolation.

In [ ]:
key_cols = ['record_id', 'pred_name']
base_status = baseline_fmd_df[key_cols + ['exact_match', 'coord_correct', 'pred_start', 'pred_end', 'truth_start', 'truth_end', 'failure_mode']].copy()
base_status = base_status.rename(columns={c: f'baseline_{c}' for c in base_status.columns if c not in key_cols})
exp_status = experiment_df[key_cols + ['exact_match', 'coord_correct', 'pred_start', 'pred_end', 'truth_start', 'truth_end', 'failure_mode', 'status']].copy()
exp_status = exp_status.rename(columns={c: f'experiment_{c}' for c in exp_status.columns if c not in key_cols})
case_compare = base_status.merge(exp_status, on=key_cols, how='outer')

fixed = case_compare[(case_compare['baseline_exact_match'] == False) & (case_compare['experiment_exact_match'] == True)].copy()
regressed = case_compare[(case_compare['baseline_exact_match'] == True) & (case_compare['experiment_exact_match'] == False)].copy()

fixed.to_csv(OUTPUT_DIR / 'fmd_fixed_cases.tsv', sep='	', index=False)
regressed.to_csv(OUTPUT_DIR / 'fmd_regressed_cases.tsv', sep='	', index=False)

print('fixed cases:', len(fixed))
print('regressed cases:', len(regressed))
display(fixed.head(30))
display(regressed.head(30))